# 2) Embeddings + Vector Database (ChromaDB)
## Corrective RAG System — Step 2 of 4

**Goal:** turn the chunks produced in notebook 1 into embeddings using Ollama (fully local, no API
key or internet required), and store them persistently in ChromaDB so the rest of the notebooks can
retrieve them without recomputing embeddings every time.

> This notebook requires [Ollama](https://ollama.com) running on your machine, with the embedding
> model pulled once: `ollama pull nomic-embed-text`

This notebook covers: *Generate embeddings and store them in a vector database.*


In [1]:
import json
import os
import urllib.request
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHUNKS_PATH = PROCESSED_DIR / "chunks.json"
VECTORSTORE_DIR = PROJECT_ROOT / "vectorstore" / "chroma_db"
COLLECTION_NAME = "crag_course_docs"
EMBEDDING_MODEL = "nomic-embed-text"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

try:
    urllib.request.urlopen(OLLAMA_BASE_URL, timeout=3)
except Exception as exc:
    raise RuntimeError(
        f"Could not reach Ollama at {OLLAMA_BASE_URL}. Make sure the Ollama app is running "
        f"(it starts automatically after installation, or run 'ollama serve' manually), and that "
        f"the '{EMBEDDING_MODEL}' model is pulled (ollama pull {EMBEDDING_MODEL})."
    ) from exc

print(f"Ollama reachable at {OLLAMA_BASE_URL}")


Ollama reachable at http://localhost:11434


## Step 1 — Load processed chunks (from notebook 01)

In [2]:
if not CHUNKS_PATH.exists():
    raise FileNotFoundError("chunks.json not found - run notebook 01_Data_Ingestion first.")

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(chunks)} chunks from {CHUNKS_PATH}")


Loaded 11 chunks from C:\Users\mahmoud\Desktop\New folder (2)\data\processed\chunks.json


## Step 2 — Build LangChain `Document` objects

In [3]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=chunk["text"],
        metadata={"source": chunk["source"], "chunk_index": chunk["chunk_index"], "id": chunk["id"]},
    )
    for chunk in chunks
]
print(f"Prepared {len(documents)} LangChain Document objects")


Prepared 11 LangChain Document objects


## Step 3 — Create the embedding model (Ollama, fully local)

In [4]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)


## Step 4 — Create / persist the Chroma vector store

We pass explicit `ids` so that re-running this cell upserts instead of duplicating the same chunks.

In [5]:
from langchain_chroma import Chroma

VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTORSTORE_DIR),
    ids=[chunk["id"] for chunk in chunks],
)
print(f"Stored {vectorstore._collection.count()} vectors in Chroma collection '{COLLECTION_NAME}'")


Stored 11 vectors in Chroma collection 'crag_course_docs'


## Step 5 — Sanity check: similarity search

In [6]:
sample_query = "What is Corrective RAG and how does it reduce hallucination?"
results = vectorstore.similarity_search_with_score(sample_query, k=4)

for doc, score in results:
    print(f"[score={score:.4f}] source={doc.metadata['source']} chunk={doc.metadata['chunk_index']}")
    print(doc.page_content[:200].replace("\n", " "), "...\n")


[score=0.5879] source=03_corrective_rag.txt chunk=0
Corrective Retrieval-Augmented Generation (CRAG)  Corrective RAG is an extension of standard Retrieval-Augmented Generation that adds a self-correction loop around the retrieval step. The core idea, i ...

[score=0.6252] source=03_corrective_rag.txt chunk=1
A typical Corrective RAG pipeline works as follows. First, the system retrieves the top-k chunks for the user's question from the vector database, exactly like standard RAG. Second, a relevance grader ...

[score=0.6284] source=01_rag_basics.txt chunk=2
The main weakness of standard RAG is that it trusts the retriever blindly. If the retrieved chunks are irrelevant, outdated, or low quality, the LLM will still try to answer using them, which often pr ...

[score=0.6331] source=03_corrective_rag.txt chunk=3
Finally, the answer-generation step uses a strict prompt that instructs the model to answer only from the verified, relevant context and to explicitly say that it does not know t

> **Note:** `persist_directory` writes the database to disk automatically, so there is no need to call `.persist()` manually — any other notebook can open the same collection from `vectorstore/chroma_db`.